# Treinamento: Meta Model (Família IV - O Juiz)
**Missão:** Consumir as probabilidades do "Alarmista" (Raios-X Globais) e do "Cético" (Magnetogramas Regionais) para tomar a decisão final de alerta de Space Weather.

**Alvo Final:** Detecção de Ameaças Severas e Catastróficas (Classes M e X).

In [ ]:
import os
import optuna
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import xgboost as xgb
from sklearn.metrics import average_precision_score, classification_report, matthews_corrcoef, f1_score
from sklearn.model_selection import GroupKFold
import joblib

load_dotenv()

## 1. Carregamento dos Dados Meta

In [ ]:
META_DIR = os.path.join(os.getenv('SLIDED_PATH'), 'meta_model')

meta_train = pd.read_parquet(os.path.join(META_DIR, 'meta_train.parquet'))
meta_test = pd.read_parquet(os.path.join(META_DIR, 'meta_test.parquet'))

print(f"Meta-Treino (Validação Original): {meta_train.shape}")
print(f"Meta-Teste (Teste Cego - Ciclo 25): {meta_test.shape}")

## 2. Preparação das Features e do Target

In [ ]:
# Features de Opinião dos Especialistas (Probabilidades e Regressões)
opinion_features = [
    'prob_gk_I', 'prob_gf_I', 'prob_s910_I', 'pred_smx_I',
    'prob_gk_III', 'prob_gf_III', 'prob_s910_III', 'pred_smx_III'
]

# Features de Contexto Básico
context_features = [
    'xrsb_flux_mean',     # Contexto Global (Onde estamos na curva?)
    'num_active_spots'    # Complexidade do Disco (Ameaça difusa?)
]

meta_features = opinion_features + context_features

def extract_xy_groups(df):
    X = df[meta_features].copy()
    y = (df['target_class'] >= 4).astype(int) # M=4, X=5
    groups = df['REGION_ID'] # Âncora de isolamento
    return X, y, groups

X_train_meta, y_train_meta, groups_train = extract_xy_groups(meta_train)
X_test_meta, y_test_meta, groups_test = extract_xy_groups(meta_test)

# TRAVA DE SEGURANÇA: Garante que as features físicas brutas não entraram
assert 'USFLUX' not in X_train_meta.columns, "ERRO: USFLUX ainda está nos dados!"

## 3. Otimização do Juiz (Optuna)
**Restrição:** O Meta Model não pode ser profundo. Forçamos `max_depth` entre 2 e 4 e alta regularização (`alpha` e `lambda`) para que ele atue apenas como uma regressão lógica sobre os modelos base.

In [ ]:
def objective(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'aucpr',
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'random_state': 42,

        'max_depth': trial.suggest_int('max_depth', 2, 4),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 15.0),

        'alpha': trial.suggest_float('alpha', 0.01, 5.0, log=True),
        'lambda': trial.suggest_float('lambda', 0.01, 5.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0)
    }

    _gkf = GroupKFold(n_splits=5)
    pr_aucs = []

    for _train_idx, _val_idx in _gkf.split(X_train_meta, y_train_meta, groups=groups_train):
        _X_tr, _y_tr = X_train_meta.iloc[_train_idx], y_train_meta.iloc[_train_idx]
        _X_va, _y_va = X_train_meta.iloc[_val_idx], y_train_meta.iloc[_val_idx]

        model = xgb.XGBClassifier(**params)
        model.fit(_X_tr, _y_tr, verbose=False)

        preds = model.predict_proba(_X_va)[:, 1]
        pr_aucs.append(average_precision_score(_y_va, preds))

    return np.mean(pr_aucs)

study = optuna.create_study(direction='maximize')
print("Iniciando treinamento do Meta Model (Validação Cruzada por Grupo)...")
study.optimize(objective, n_trials=50)
print(f"\nBest Meta PR-AUC Honesto: {study.best_value:.4f}")

## 4. Treinamento Final e Threshold Tuning

In [ ]:
best_params = study.best_params
best_params.update({'objective': 'binary:logistic', 'random_state': 42})

# =======================================================
# 1. LOOP MANUAL DE OUT-OF-FOLD (Totalmente a prova de falhas)
# =======================================================
# Vetor vazio para guardar as predições de cada mancha no seu momento de Validação
oof_preds = np.zeros(len(X_train_meta))
gkf = GroupKFold(n_splits=5)

for train_idx, val_idx in gkf.split(X_train_meta, y_train_meta, groups=groups_train):
    # Separação estrita garantida pelo GroupKFold
    X_tr, y_tr = X_train_meta.iloc[train_idx], y_train_meta.iloc[train_idx]
    X_va, y_va = X_train_meta.iloc[val_idx], y_train_meta.iloc[val_idx]

    # Treina o modelo cego para as manchas de validação
    meta_model_cv = xgb.XGBClassifier(**best_params)
    meta_model_cv.fit(X_tr, y_tr, verbose=False)

    # Salva a probabilidade honesta nos índices originais do DataFrame
    oof_preds[val_idx] = meta_model_cv.predict_proba(X_va)[:, 1]


# =======================================================
# 2. CALIBRAÇÃO DE THRESHOLD HONESTO
# =======================================================
from sklearn.metrics import precision_recall_curve
precisions, recalls, thresholds = precision_recall_curve(y_train_meta, oof_preds)

# Acha o ponto de equilíbrio natural (F1-Score Máximo)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores)
optimal_meta_threshold = thresholds[best_idx]

print(f"Meta Threshold Honesto Ajustado para: {optimal_meta_threshold:.4f}")


# =======================================================
# 3. TREINAMENTO FINAL PARA PRODUÇÃO
# =======================================================
# Com o limiar cravado honestamente, o modelo final pode ver todos os dados
meta_model = xgb.XGBClassifier(**best_params)
meta_model.fit(X_train_meta, y_train_meta, verbose=False)

## 5. A Prova Final (Desempenho no Ciclo Solar 25)

In [ ]:
y_prob_test = meta_model.predict_proba(X_test_meta)[:, 1]
y_pred_test = (y_prob_test >= optimal_meta_threshold).astype(int)

print("--- AVALIAÇÃO DO META MODEL (CICLO 25) ---")
print(classification_report(y_test_meta, y_pred_test, target_names=['Moderado (< M)', 'Severo (M/X)']))

mcc = matthews_corrcoef(y_test_meta, y_pred_test)
pr_auc = average_precision_score(y_test_meta, y_prob_test)
f1 = f1_score(y_test_meta, y_pred_test)

print(f"\nMCC: {mcc:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")
print(f"F1-Score: {f1:.4f}")

## 6. Interpretabilidade: O que o Juiz aprendeu?

In [ ]:
importance_df = pd.DataFrame({
    'Feature': X_train_meta.columns,
    'Importance (Gain)': meta_model.feature_importances_
}).sort_values(by='Importance (Gain)', ascending=False)

importance_df = importance_df.reset_index(drop=True)

display(importance_df)

## 7. Exportação

In [ ]:
META_EXPORT_PATH = os.path.join(os.getenv('SLIDED_PATH'), 'meta_model', 'meta_model_v1.joblib')
joblib.dump({'model': meta_model, 'threshold': optimal_meta_threshold}, META_EXPORT_PATH)
print(f"O Juiz foi exportado com sucesso para: {META_EXPORT_PATH}")